# Points PMF

Integer points probabilities for one player-game. **A** draws minutes and rate independently. **B** draws a stored out-of-sample PIT pair from the empirical copula. Both push those uniforms through that row's own quantile functions. The PMF is `k = clip(floor(minutes * rate + 0.5), 0, 90)` from 10,000 draws.

Rows are the joint out-of-sample table from the points notebook (`data/oos/nba/minutes_rate_oos.parquet`). Minutes tiers are predicted minutes q50: `<15`, `15-24`, `24-31`, `31+`.

Dev is where the report is read. It is the last pre-holdout season's OOS rows (2024-25). For each dev row, B's pool is pairs with `game_date` before that row and `include_holdout=False`. Holdout is 2025-26. It is scored once, after the draw count, seed, and tolerances above are fixed. B's holdout pool is every pre-holdout pair (`include_holdout=False`).

Each pseudo-line is its own sample of player-games. Five lines from one player-game are not five observations.

Flags require both a tolerance miss and a gap of more than 2 SE:

| Check | Tolerance | Ideal |
|---|---:|---:|
| PIT mean | ±0.01 | 0.50 |
| 12 × PIT variance | ±0.05 | 1 |
| PIT share below 0.05 or above 0.95 | ±1 percentage point | 5% |
| Predicted vs observed rate (over, favored side, P(points = 0)) | ±1.5 percentage points | predicted rate |

Central 50/80/90% PIT coverage is reported with its SE and is not in that flag list. Log score is higher-better; RPS is lower-better. Paired differences are B minus A, with a 95% interval from 2,000 resamples of game dates.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path("/Users/alexgonzalez/Documents/nba_quant")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from models.shared.copula import (
    EmpiricalCopula,
    IndependentCopula,
    build_pit_pairs,
    minutes_tier,
)
from models.shared.metrics import (
    FLAG_SE,
    N_BOOT_DATES,
    OVER_RATE_TOLERANCE,
    PIT_MEAN_TOLERANCE,
    PIT_TAIL_TOLERANCE,
    PIT_VAR_SCALE_TOLERANCE,
    evaluate_points_pmfs,
)
from models.shared.minutes_sampler import canonical_id
from models.shared.minutes_sampler import load_tail_sidecar as load_minutes_tails
from models.shared.ppm_sampler import load_tail_sidecar as load_rate_tails
from models.shared.simulate import points_pmf
from src.features.minutes.rolling import prior_ewm, prior_expanding, prior_shift

SEED = 42
N_DRAWS = 10_000
KMAX = 90
HOLDOUT_SEASON = "2025-26"
LAST_PREHOLDOUT_SEASON = "2024-25"
OOS_PATH = ROOT / "data" / "oos" / "nba" / "minutes_rate_oos.parquet"
MINUTES_TAILS = ROOT / "models" / "saved_models" / "min_nba_tails_2026-04-12.joblib"
RATE_TAILS = ROOT / "models" / "saved_models" / "pts_nba_tails_2026-04-12.joblib"
SEASONS = [
    "2019-20",
    "2020-21",
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26",
]

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)
print(
    f"Flag when the gap exceeds the tolerance "
    f"(PIT mean ±{PIT_MEAN_TOLERANCE}, 12*Var ±{PIT_VAR_SCALE_TOLERANCE}, "
    f"tails ±{PIT_TAIL_TOLERANCE:.0%}, rates ±{OVER_RATE_TOLERANCE:.1%}) "
    f"and {FLAG_SE:g} SE."
)


Flag when the gap exceeds the tolerance (PIT mean ±0.01, 12*Var ±0.05, tails ±1%, rates ±1.5%) and 2 SE.


In [2]:
def present(title, frame):
    print(f"\n{title}")
    if frame is None or frame.empty:
        print("no rows")
        return
    shown = frame.copy()
    for column in shown.columns:
        if pd.api.types.is_float_dtype(shown[column]):
            shown[column] = shown[column].round(4)
    display(shown)
    flag_columns = [column for column in frame.columns if str(column).endswith("flag")]
    if not flag_columns:
        return
    mask = np.zeros(len(frame), dtype=bool)
    for column in flag_columns:
        mask |= frame[column].fillna(False).to_numpy(dtype=bool)
    print(f"flagged {int(mask.sum())} of {len(frame)}")
    if mask.any():
        display(shown.loc[mask])


def histogram_table(frame):
    labeled = frame.copy()
    labeled["bin"] = [
        f"{lo:.1f}-{hi:.1f}" for lo, hi in zip(labeled["lo"], labeled["hi"], strict=True)
    ]
    wide = labeled.pivot(index=["model", "tier"], columns="bin", values="share")
    return wide.reset_index()


def distribution(rows, copula):
    pmf, _mean, _median = points_pmf(
        rows,
        copula,
        n_draws=N_DRAWS,
        kmax=KMAX,
        minutes_tables=minutes_tables,
        rate_tables=rate_tables,
        seed=SEED,
    )
    return pmf


def independent_distribution(rows):
    copula = IndependentCopula(
        pairs,
        before_date=rows["game_date"].min(),
        include_holdout=False,
    )
    return distribution(rows, copula)


def empirical_distribution(rows, *, before_date=None):
    """One copula per date when `before_date` is None. Otherwise one pool."""
    if before_date is not None:
        copula = EmpiricalCopula(pairs, before_date=before_date, include_holdout=False)
        return distribution(rows, copula)
    pieces = []
    labels = []
    grouped = list(rows.groupby("game_date", sort=True).groups.items())
    for step, (date, index) in enumerate(grouped, start=1):
        copula = EmpiricalCopula(pairs, before_date=date, include_holdout=False)
        pieces.append(distribution(rows.loc[index], copula))
        labels.append(np.asarray(index))
        if step == 1 or step % 20 == 0 or step == len(grouped):
            print(f"  empirical dates {step}/{len(grouped)}")
    stacked = np.concatenate(pieces, axis=0)
    order = np.concatenate(labels)
    position = pd.Series(np.arange(stacked.shape[0]), index=order)
    return stacked[position.loc[rows.index].to_numpy()]


In [3]:
panel_columns = [
    "season_year",
    "game_id",
    "player_id",
    "game_date",
    "pts",
    "minutes",
    "start_position",
]
panel = pd.concat(
    [
        pd.read_parquet(
            ROOT / "data" / "silver" / "nba" / season / "regular_season" / "player_gamelogs.parquet",
            columns=panel_columns,
        )
        for season in SEASONS
    ],
    ignore_index=True,
)
panel = panel.loc[panel["minutes"].astype(float) > 0].copy()
panel["pts"] = panel["pts"].astype(float)
panel["game_date"] = pd.to_datetime(panel["game_date"])
panel = panel.sort_values(
    ["game_date", "game_id", "player_id"], kind="mergesort"
).reset_index(drop=True)
panel["starting"] = (
    panel["start_position"].astype("string").str.strip().isin(["G", "F", "C"]).astype(int)
)
panel["pts_lag_1"] = prior_shift(panel, "pts", ["player_id"])
panel["pts_ewm_hl_3"] = prior_ewm(panel, "pts", ["player_id"], halflife=3)
panel["season_pts_mean"] = prior_expanding(
    panel, "pts", ["player_id", "season_year"], "mean"
)
panel["game_key"] = panel["game_id"].map(canonical_id)
panel["player_key"] = panel["player_id"].map(canonical_id)
if panel.duplicated(["game_key", "player_key"]).any():
    raise AssertionError("panel player-games are not unique")

oos = pd.read_parquet(OOS_PATH)
oos["game_date"] = pd.to_datetime(oos["game_date"])
oos["game_key"] = oos["game_id"].map(canonical_id)
oos["player_key"] = oos["player_id"].map(canonical_id)
baseline_columns = [
    "game_key",
    "player_key",
    "season_year",
    "starting",
    "pts_lag_1",
    "pts_ewm_hl_3",
    "season_pts_mean",
]
aligned = oos.merge(
    panel[baseline_columns],
    on=["game_key", "player_key"],
    how="left",
    validate="one_to_one",
)
if aligned["starting"].isna().any():
    raise AssertionError("OOS rows are missing a starter flag")
if (aligned["minutes"].to_numpy(dtype=float) <= 0).any():
    raise AssertionError("OOS rows include 0 minutes")
aligned["tier"] = minutes_tier(aligned["minutes_q_0.50"])

minutes_tables = load_minutes_tails(MINUTES_TAILS)
rate_tables = load_rate_tails(RATE_TAILS)
print(f"Building PIT pairs for {len(aligned):,} OOS rows")
pairs = build_pit_pairs(
    aligned,
    minutes_tables=minutes_tables,
    rate_tables=rate_tables,
    seed=SEED,
)

pre = aligned.loc[~aligned["is_holdout"].astype(bool)].copy()
last_season = pre.groupby("season_year", sort=False)["game_date"].max().idxmax()
if last_season != LAST_PREHOLDOUT_SEASON:
    raise AssertionError(f"last pre-holdout season is {last_season}")
dev = pre.loc[pre["season_year"].eq(last_season)].reset_index(drop=True)
holdout = aligned.loc[aligned["is_holdout"].astype(bool)].reset_index(drop=True)
if set(holdout["season_year"].astype(str)) != {HOLDOUT_SEASON}:
    raise AssertionError("holdout season is not 2025-26")
earlier = pre.loc[~pre["season_year"].eq(last_season)]
if earlier["game_date"].max() >= dev["game_date"].min():
    raise AssertionError("dev season overlaps earlier OOS rows")
if holdout["game_date"].min() <= pre["game_date"].max():
    raise AssertionError("holdout dates overlap pre-holdout")

print(f"OOS rows {len(aligned):,}")
print(
    f"Dev {last_season}: {len(dev):,} rows, "
    f"{dev['game_date'].min().date()} → {dev['game_date'].max().date()}, "
    f"{dev['game_date'].nunique()} dates"
)
print(
    f"Holdout {HOLDOUT_SEASON}: {len(holdout):,} rows, "
    f"{holdout['game_date'].min().date()} → {holdout['game_date'].max().date()}"
)
print("Holdout scores are in the last section.")


Building PIT pairs for 87,091 OOS rows


KeyboardInterrupt: 

## Dev

2024-25 OOS rows only. B is refit at each game date: `before_date` is that date, and holdout pairs stay out. A is an independent copula with the same as-of rule. Nothing in this section is chosen from 2025-26 results.

`var_scale` is 12 times the unbiased PIT variance. `gap_b` is B's mean P(Over) minus the observed over-rate. `observed_se` is the binomial SE of that over-rate. The flag uses `gap_se`, the SE of the paired gap, and the 1.5 percentage-point tolerance.

On the naive points table, the PMF row's MAE is the PMF median and its RMSE is the PMF mean. Last game, season-to-date mean, and EWMA halflife 3 are points, from appearances only. At L0, the constant over-rate is the tier's historical rate of going over its own EWMA line, fit on OOS rows before the first dev date. The slope regresses `(over - base rate)` on `(B's P(Over) - base rate)`. Near 0, B adds nothing past that naive line.


In [ ]:
dev_pmf_a = independent_distribution(dev)
dev_pmf_b = empirical_distribution(dev)
print(f"Dev PMFs {dev_pmf_a.shape[0]:,} x {dev_pmf_a.shape[1] - 1}")


In [ ]:
dev_report = evaluate_points_pmfs(
    dev,
    dev_pmf_a,
    dev_pmf_b,
    history=aligned,
    n_boot=N_BOOT_DATES,
    seed=SEED,
)
pre_dev = aligned.loc[aligned["game_date"] < dev["game_date"].min()]
dev_fit = int(
    dev_report["naive_line"].loc[dev_report["naive_line"]["tier"].eq("all"), "n_fit"].iloc[0]
)
if dev_fit > len(pre_dev):
    raise AssertionError("dev base rate saw rows from the scored period or later")
print(f"L0 base rate fit on {dev_fit:,} rows before {dev['game_date'].min().date()}")

present("Dev PIT", dev_report["pit"])
present("Dev PIT histogram, ideal share 0.1", histogram_table(dev_report["pit_histogram"]))
present("Dev P(points = 0)", dev_report["zero"])
present("Dev log score and RPS (diff = B - A)", dev_report["scores"])
present("Dev pseudo-lines", dev_report["lines"])
present("Dev logistic calibration at L0 (ideal intercept 0, slope 1)", dev_report["calibration"])
present("Dev favored side at L0", dev_report["favored"])
present("Dev naive points", dev_report["naive_points"])
present("Dev naive line at L0", dev_report["naive_line"])


## Holdout 2025-26

Draw count, seed, tolerances, and the dev report stay as they were. B uses one pool: every pre-holdout PIT pair, `include_holdout=False`. The L0 constant is fit on all pre-holdout OOS rows, including the dev season, and not on holdout rows.


In [ ]:
holdout_cutoff = holdout["game_date"].min()
holdout_pool = EmpiricalCopula(
    pairs, before_date=holdout_cutoff, include_holdout=False
)
pre_pairs = int((~pairs["is_holdout"].astype(bool)).sum())
if len(holdout_pool.pairs) != pre_pairs:
    raise AssertionError("holdout copula is not the full pre-holdout pool")
print(
    f"Holdout B pool: {len(holdout_pool.pairs):,} pre-holdout pairs "
    f"before {holdout_cutoff.date()}"
)

holdout_pmf_a = independent_distribution(holdout)
holdout_pmf_b = empirical_distribution(holdout, before_date=holdout_cutoff)
print(f"Holdout PMFs {holdout_pmf_a.shape[0]:,} x {holdout_pmf_a.shape[1] - 1}")


In [ ]:
holdout_report = evaluate_points_pmfs(
    holdout,
    holdout_pmf_a,
    holdout_pmf_b,
    history=aligned,
    n_boot=N_BOOT_DATES,
    seed=SEED,
)
pre_holdout = aligned.loc[~aligned["is_holdout"].astype(bool)]
holdout_fit = int(
    holdout_report["naive_line"]
    .loc[holdout_report["naive_line"]["tier"].eq("all"), "n_fit"]
    .iloc[0]
)
if holdout_fit > len(pre_holdout):
    raise AssertionError("holdout base rate used holdout rows")
if holdout_fit <= dev_fit:
    raise AssertionError("holdout base rate dropped the dev season")
print(f"L0 base rate fit on {holdout_fit:,} pre-holdout rows")

present("Holdout PIT", holdout_report["pit"])
present("Holdout PIT histogram, ideal share 0.1", histogram_table(holdout_report["pit_histogram"]))
present("Holdout P(points = 0)", holdout_report["zero"])
present("Holdout log score and RPS (diff = B - A)", holdout_report["scores"])
present("Holdout pseudo-lines", holdout_report["lines"])
present(
    "Holdout logistic calibration at L0 (ideal intercept 0, slope 1)",
    holdout_report["calibration"],
)
present("Holdout favored side at L0", holdout_report["favored"])
present("Holdout naive points", holdout_report["naive_points"])
present("Holdout naive line at L0", holdout_report["naive_line"])
